In [ ]:
import sys
import os

# Set project root relative to notebook location
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Losses import PricingLoss

: 

In [ ]:
import numpy as np
import mbtr.utils as ut
from scipy.linalg import hankel
from scipy.stats import expon
import matplotlib.pyplot as plt
from mbtr.mbtr import MBT
from mbtr.utils import set_figure
from mbtr import LOSS_MAP
from Losses.PricingLoss import PricingLikelihood

LOSS_MAP['pricing'] = PricingLikelihood # this line has to be added for the MBT class to recognize the PricingLikelihood subclass as a type of loss

In [ ]:
from Data.pg15_cleaning import pg15_training

joint_severity = pg15_training[(pg15_training['AvIndtppd'] != 0) & (pg15_training['AvIndtpbi'] != 0)]
material_damage = joint_severity['AvIndtppd']
bodily_injury = joint_severity['AvIndtpbi']

plt.scatter(material_damage, bodily_injury, alpha = 0.3)
plt.show()

In [ ]:
import scipy.stats as stats

def rank_pseudo_obs(y):
        """
        Return rank-based pseudo-observations from initial target matrix

        :param y: observation matrix (n_obs, 2)

        :return: rank-based pseudo-observations matrix (n_obs, 2)
        """
        n_obs = y.shape[0]
        u1 = stats.rankdata(y[:, 0])/(n_obs + 1)
        u2 = stats.rankdata(y[:, 1])/(n_obs + 1)
        return np.hstack((u1.reshape(-1,1), u2.reshape(-1,1)))

sev_array = joint_severity[['AvIndtppd', 'AvIndtpbi']].to_numpy()
residuals = rank_pseudo_obs(sev_array)

print(stats.kendalltau(residuals[:, 0], residuals[:, 1]))
plt.scatter(residuals[:, 0], residuals[:, 1], alpha = 0.3)
plt.show()

In [ ]:
plt.hist(pg15_training['TotalNum'])
plt.show()

In [ ]:
covariates = pg15_training.loc[ : , ~pg15_training.columns.isin(['PolNum', 'CalYear', 'SubGroup2', 'Exppdays', 'Numtppd', 'Numtpbi', 'Indtppd', 'Indtpbi', 'TotalNum', 'Exposure', 'AvIndtppd', 'AvIndtpbi'])]
output = pg15_training[['TotalNum', 'Numtppd', 'AvIndtppd', 'AvIndtpbi', 'Exposure']]

x = covariates.to_numpy()
y = output.to_numpy()

In [ ]:
m = MBT(n_boosts = 10, learning_rate = 0.3, loss_type = "pricing", refit = False, warmup = 5).fit(x, y, do_plot = False)